In [ ]:
import os
import re
import sys
import subprocess
from pathlib import Path

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    pass
if not IN_COLAB and os.path.isdir("/content"):
    IN_COLAB = True


def _pip_install_requirements(req: Path, *, skip_torch: bool) -> None:
    if not req.is_file():
        return
    if skip_torch:
        kept = []
        for line in req.read_text().splitlines():
            s = line.split("#", 1)[0].strip()
            if not s:
                continue
            if re.match(r"(?i)^torch\b", s):
                continue
            kept.append(line)
        tmp = Path("/tmp/stk_mat2011_requirements_no_torch.txt")
        tmp.write_text("\n".join(kept) + "\n")
        target = tmp
        print("Colab: pip installing requirements (torch line skipped)")
    else:
        target = req
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(target)], check=False)


REPO_URL = "https://github.com/egil10/stk-mat2011.git"
if IN_COLAB:
    REPO_DIR = Path("/content/stk-mat2011")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=False)
    _pip_install_requirements(REPO_DIR / "requirements.txt", skip_torch=True)
    _nb = REPO_DIR / "code" / "notebooks"
    if _nb.is_dir():
        os.chdir(str(_nb))
    print("Working directory:", os.getcwd())
else:
    REPO_DIR = Path.cwd().resolve().parents[1]
    _pip_install_requirements(REPO_DIR / "requirements.txt", skip_torch=False)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "statsmodels"], check=False)

In [ ]:
import importlib.util
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from scipy import stats
from torch import optim

import statsmodels.api as sm

SCRIPTS_DIR = (REPO_DIR / "code" / "scripts").resolve()
_nn = SCRIPTS_DIR / "nnhmm.py"
if not _nn.is_file():
    raise FileNotFoundError(f"nnhmm.py not found at {_nn}")

_spec = importlib.util.spec_from_file_location("nnhmm", str(_nn))
_nnhmm_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_nnhmm_mod)
NNHMM = _nnhmm_mod.NNHMM

In [ ]:
def interval_score_2sided(y, lo, hi, alpha=0.05):
    """Per-observation interval score; lower is better."""
    y = np.asarray(y, dtype=float)
    lo = np.asarray(lo, dtype=float)
    hi = np.asarray(hi, dtype=float)
    width = hi - lo
    below = y < lo
    above = y > hi
    penalty_below = (lo - y) * below
    penalty_above = (y - hi) * above
    return width + (2.0 / alpha) * (penalty_below + penalty_above)


def crps_gaussian(y, mean, std):
    """
    CRPS for N(mean, std^2), each time point independent.
    y, mean, std: same shape.
    """
    y = np.asarray(y, dtype=float)
    mean = np.asarray(mean, dtype=float)
    std = np.asarray(std, dtype=float)
    std = np.maximum(std, 1e-12)
    z = (y - mean) / std
    return std * (z * (2.0 * stats.norm.cdf(z) - 1.0) + 2.0 * stats.norm.pdf(z) - 1.0 / np.sqrt(np.pi))


def summarize_forecast(name, y_true, pred_mean, lo, hi, alpha):
    y_true = np.asarray(y_true, dtype=float)
    pred_mean = np.asarray(pred_mean, dtype=float)
    lo = np.asarray(lo, dtype=float)
    hi = np.asarray(hi, dtype=float)
    ints = interval_score_2sided(y_true, lo, hi, alpha=alpha)
    row = {
        "model": name,
        "rmse": float(np.sqrt(np.mean((y_true - pred_mean) ** 2))),
        "coverage": float(np.mean((y_true >= lo) & (y_true <= hi))),
        "mean_width": float(np.mean(hi - lo)),
        "interval_score_mean": float(np.mean(ints)),
    }
    return row, ints


def diebold_mariano_hac(loss_a, loss_b, maxlags=10):
    """
    HAC t-test for E[d_t]=0 where d_t = loss_a - loss_b (loss = per-time score).
    """
    d = np.asarray(loss_a, dtype=float) - np.asarray(loss_b, dtype=float)
    n = len(d)
    res = sm.OLS(d, np.ones(n)).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags})
    tval = float(res.tvalues[0])
    pval = float(res.pvalues[0])
    return {"mean_d": float(d.mean()), "t_hac": tval, "p_two_sided": pval, "n": n}


def pit_gaussian(y, mean, std):
    """PIT = Phi((y-m)/s) for Gaussian predictive; uniform under calibration."""
    std = np.maximum(np.asarray(std, dtype=float), 1e-12)
    return stats.norm.cdf((np.asarray(y, dtype=float) - mean) / std)

In [ ]:
SEED = 42
K = 2

SYMBOL = "eurusd"
MODEL_MAX_POINTS = 200_000
MODEL_STRIDE_MANUAL = None
CLIP_Q = (0.001, 0.999)
DROP_ZERO_RETURNS = True

EPOCHS = 400
PATIENCE = 30
LR = 1e-2

ALPHA = 0.05
Z95 = stats.norm.ppf(1 - ALPHA / 2)

PLOT_N = 4000
DM_MAXLAGS = 10

# Optional: set path to skip training
CHECKPOINT_PATH = None  # e.g. Path(REPO_DIR / "code/notebooks/hw4_nnhmm.pt")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
try:
    from google.colab import drive
    _colab = True
except Exception:
    _colab = False

if _colab:
    drive.mount("/content/drive", force_remount=True)
    OUT_BASES = [
        Path("/content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed_mega"),
        Path("/content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed"),
        Path("/content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data"),
    ]
else:
    OUT_BASES = [Path("../data/processed_mega"), Path("../data/processed")]

patterns = [
    f"*{SYMBOL}*master*.parquet",
    f"*mega_master*.parquet",
    "*master_returns*.parquet",
    "*master*.parquet",
]

candidates = []
for base in OUT_BASES:
    if not base.exists():
        continue
    for pat in patterns:
        for p in base.glob(pat):
            if "daily_counts" in p.name:
                continue
            candidates.append((p.stat().st_mtime, p))

if not candidates:
    raise FileNotFoundError("No master parquet found.")

_, data_path = sorted(candidates, key=lambda x: x[0], reverse=True)[0]
print("Using master file:", data_path)

df = pd.read_parquet(data_path)
df = df.dropna(subset=["returns", "datetime"]).copy()
df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
df = df.dropna(subset=["datetime"]).sort_values("datetime").reset_index(drop=True)

if DROP_ZERO_RETURNS:
    df = df[df["returns"] != 0].copy()

if CLIP_Q is not None:
    lo, hi = df["returns"].quantile(CLIP_Q[0]), df["returns"].quantile(CLIP_Q[1])
    df["returns"] = df["returns"].clip(lo, hi)

if MODEL_STRIDE_MANUAL is not None:
    stride = int(MODEL_STRIDE_MANUAL)
else:
    stride = 1
    if len(df) > MODEL_MAX_POINTS:
        stride = int(np.ceil(len(df) / MODEL_MAX_POINTS))

print(f"Original points: {len(df):,} | stride={stride}")
if stride > 1:
    df = df.iloc[::stride].copy().reset_index(drop=True)

n = len(df)
i1, i2 = int(0.70 * n), int(0.85 * n)
df_tr = df.iloc[:i1].copy()
df_va = df.iloc[i1:i2].copy()
df_te = df.iloc[i2:].copy()

print("train:", len(df_tr), "val:", len(df_va), "test:", len(df_te))
print("test range:", df_te["datetime"].min(), "->", df_te["datetime"].max())

In [ ]:
def to_tensor(x):
    return torch.tensor(x.values, dtype=torch.float32).unsqueeze(1).to(device)


y_tr = to_tensor(df_tr["returns"])
y_va = to_tensor(df_va["returns"])
y_te = to_tensor(df_te["returns"])

model = NNHMM(n_states=K, input_dim=1).to(device)

if CHECKPOINT_PATH is not None and Path(CHECKPOINT_PATH).is_file():
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
    model.eval()
    print("Loaded checkpoint:", CHECKPOINT_PATH)
else:
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=10
    )

    best_val = np.inf
    best_state = None
    bad = 0
    hist_tr, hist_va = [], []

    for ep in range(EPOCHS):
        model.train()
        optimizer.zero_grad()
        loss_tr = -model(y_tr) / len(y_tr)
        loss_tr.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            loss_va = -model(y_va) / len(y_va)

        scheduler.step(loss_va)
        tr = float(loss_tr.item())
        va = float(loss_va.item())
        hist_tr.append(tr)
        hist_va.append(va)

        if va < best_val - 1e-6:
            best_val = va
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1

        if bad >= PATIENCE:
            print(f"early stop at ep={ep}, best_val={best_val:.6f}")
            break

    model.load_state_dict(best_state)
    print("best val NLL per sample:", best_val)

    if CHECKPOINT_PATH is not None:
        Path(CHECKPOINT_PATH).parent.mkdir(parents=True, exist_ok=True)
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print("Saved:", CHECKPOINT_PATH)

In [ ]:
r_tr = df_tr["returns"].values
r_va = df_va["returns"].values
r_te = df_te["returns"].values
true = r_te

# ----- AR(1) -----
X = np.column_stack([np.ones(len(r_tr) - 1), r_tr[:-1]])
y_ = r_tr[1:]
a_hat, b_hat = np.linalg.lstsq(X, y_, rcond=None)[0]
prev_te = np.concatenate([r_va[-1:], r_te[:-1]])
pred_ar1 = a_hat + b_hat * prev_te
res_tr = y_ - (a_hat + b_hat * r_tr[:-1])
s_hat = float(res_tr.std(ddof=1))
lo_ar1 = pred_ar1 - Z95 * s_hat
hi_ar1 = pred_ar1 + Z95 * s_hat
std_ar1 = np.full_like(true, s_hat)

# ----- Naive (iid train marginal) -----
s0 = float(r_tr.std(ddof=1))
pred_naive = np.zeros_like(true)
lo_naive = np.full_like(true, -Z95 * s0)
hi_naive = np.full_like(true, Z95 * s0)
std_naive = np.full_like(true, s0)

# ----- NNHMM -----
model.eval()
with torch.no_grad():
    nll_te = float((-model(y_te) / len(y_te)).item())
    y_all = torch.cat([y_tr, y_va, y_te], dim=0)
    N_tr, N_va, N_te = len(y_tr), len(y_va), len(y_te)
    test_start = N_tr + N_va

    log_alpha = model.forward_log_alpha(y_all)
    log_norm = torch.logsumexp(log_alpha, dim=1, keepdim=True)
    filt = torch.exp(log_alpha - log_norm)

    filt_slice = filt[test_start - 1 : test_start - 1 + N_te]
    trans = model.transition_matrix.detach()
    mu = model.means.detach().view(K)
    sigma = torch.exp(model.log_std.detach()).view(K)

    state_pred = filt_slice @ trans
    mix_mean = state_pred @ mu
    mix_var = state_pred @ (sigma**2) + state_pred @ (mu**2) - mix_mean**2
    mix_std = torch.sqrt(torch.clamp(mix_var, min=1e-12))
    lo_mix = mix_mean - Z95 * mix_std
    hi_mix = mix_mean + Z95 * mix_std

    k_star = torch.argmax(state_pred, dim=1)
    hard_mean = mu[k_star]
    hard_std = sigma[k_star]
    lo_hard = hard_mean - Z95 * hard_std
    hi_hard = hard_mean + Z95 * hard_std

mix_mean_np = mix_mean.cpu().numpy()
lo_mix_np = lo_mix.cpu().numpy()
hi_mix_np = hi_mix.cpu().numpy()
std_mix_np = mix_std.cpu().numpy()

hard_mean_np = hard_mean.cpu().numpy()
lo_hard_np = lo_hard.cpu().numpy()
hi_hard_np = hi_hard.cpu().numpy()
std_hard_np = hard_std.cpu().numpy()

print("NNHMM test NLL per sample:", nll_te)

In [ ]:
rows = []
losses = {}

for name, pm, lo, hi in [
    ("AR(1)", pred_ar1, lo_ar1, hi_ar1),
    ("NNHMM_mixture", mix_mean_np, lo_mix_np, hi_mix_np),
    ("NNHMM_hard", hard_mean_np, lo_hard_np, hi_hard_np),
    ("Naive_iid", pred_naive, lo_naive, hi_naive),
]:
    row, L = summarize_forecast(name, true, pm, lo, hi, ALPHA)
    rows.append(row)
    losses[name] = L

crps_rows = []
crps_rows.append({"model": "AR(1)", "crps_mean": float(crps_gaussian(true, pred_ar1, std_ar1).mean())})
crps_rows.append({"model": "NNHMM_hard", "crps_mean": float(crps_gaussian(true, hard_mean_np, std_hard_np).mean())})
crps_rows.append({"model": "Naive_iid", "crps_mean": float(crps_gaussian(true, pred_naive, std_naive).mean())})
crps_rows.append({"model": "NNHMM_mixture", "crps_mean": np.nan})  # not Gaussian; leave NaN or add MC later

board = pd.DataFrame(rows).merge(pd.DataFrame(crps_rows), on="model", how="left")
board = board.sort_values("interval_score_mean").reset_index(drop=True)
display(board)

losses_df = pd.DataFrame(losses)

In [ ]:
pairs = [
    ("NNHMM_mixture", "AR(1)"),
    ("NNHMM_hard", "AR(1)"),
    ("NNHMM_mixture", "Naive_iid"),
]

dm_results = []
for a, b in pairs:
    out = diebold_mariano_hac(losses[a], losses[b], maxlags=DM_MAXLAGS)
    out["comparison"] = f"{a} minus {b} (loss)"
    dm_results.append(out)

display(pd.DataFrame(dm_results))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, (title, pit) in zip(
    axes,
    [
        ("AR(1)", pit_gaussian(true, pred_ar1, std_ar1)),
        ("NNHMM hard", pit_gaussian(true, hard_mean_np, std_hard_np)),
        ("Naive", pit_gaussian(true, pred_naive, std_naive)),
    ],
):
    ax.hist(pit, bins=20, density=True, color="steelblue", alpha=0.85)
    ax.axhline(1.0, color="black", ls="--", lw=1)
    ax.set_title(title)
    ax.set_xlim(0, 1)
plt.suptitle("PIT histograms (Gaussian models, test)")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
models = board["model"].tolist()
axes[0].bar(models, board["coverage"], color="seagreen", alpha=0.85)
axes[0].axhline(1 - ALPHA, color="black", ls="--", label="nominal")
axes[0].set_ylabel("coverage")
axes[0].set_title("Empirical 95% coverage")
axes[0].tick_params(axis="x", rotation=25)
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

axes[1].bar(models, board["interval_score_mean"], color="coral", alpha=0.85)
axes[1].set_ylabel("mean interval score")
axes[1].set_title("Interval score (lower is better)")
axes[1].tick_params(axis="x", rotation=25)
axes[1].grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

N = min(PLOT_N, len(df_te))
x = df_te.iloc[-N:]["datetime"].values
yt = df_te.iloc[-N:]["returns"].values
plt.figure(figsize=(14, 4))
plt.plot(x, yt, color="black", lw=0.7, label="true")
plt.plot(x, mix_mean_np[-N:], color="blue", lw=1.0, label="NNHMM mix mean")
plt.fill_between(x, lo_mix_np[-N:], hi_mix_np[-N:], color="dodgerblue", alpha=0.2, label="95% mix")
plt.title("Test tail — NNHMM mixture")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
out_path = Path("hw4_scoring_summary.json")
payload = {
    "data_path": str(data_path),
    "n_train": len(df_tr),
    "n_val": len(df_va),
    "n_test": len(df_te),
    "alpha": ALPHA,
    "nnhmm_test_nll_per_sample": nll_te,
    "scoreboard": board.to_dict(orient="records"),
    "diebold_mariano": dm_results,
}
out_path.write_text(json.dumps(payload, indent=2))
print("Wrote", out_path.resolve())